# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AlishaYaqub/FlyRank-ML-internship-week1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [25]:
%pip -q install duckdb
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row means one page on one day. Table used: fact_content_daily_performance. Time window: March 2026, a mid panel month. Not using June 2026, since it is the last month with no future to check against.

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

con.sql("""
    SELECT COUNT(*) AS rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬────────────┬────────────┐
│  rows   │  min_date  │  max_date  │
│  int64  │    date    │    date    │
├─────────┼────────────┼────────────┤
│ 9841378 │ 2026-03-01 │ 2026-03-31 │
└─────────┴────────────┴────────────┘



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature columns: gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions, ga4_engaged_sessions, sessions_organic, scroll_events. These are daily signals I would know before making a prediction.

Label: not a single column here. I will build it myself by comparing a page's performance in an early part of March against a later part, since no ready made trend column exists in this table.

Context: report_date, client_hash_id, content_hash_id, month, and the four availability flags (client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available). Used only for filtering and grouping, never as model input, since the ids are just pseudonyms and the flags just tell me if data exists.

Excluded: sessions_paid, sessions_referral, sessions_social. My lane is about organic search decline, so these traffic sources are outside what I am predicting.

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

con.sql("""
    DESCRIBE SELECT * FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*


I am checking three things about my March slice. First, grain: does one page ever appear twice on the same day. Second, size: how many rows exist and what dates they cover. Third, availability: how many rows actually have real search data, using the gsc_data_available flag with IS TRUE, since a missing flag does not mean zero activity, it means no data was collected.

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Query 1: grain check, one page should never appear twice on the same day
print("Grain check (should be empty):")
con.sql("""
    SELECT content_hash_id, report_date, COUNT(*) AS n
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY content_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").show()

# Query 2: row count and date span for the March slice
print("Row count and date span:")
con.sql("""
    SELECT COUNT(*) AS rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").show()

# Query 3: availability, how many rows actually have usable search data
print("Availability check:")
con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_with_gsc_data
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").show()

Grain check (should be empty):


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬─────────────┬───────┐
│ content_hash_id │ report_date │   n   │
│     varchar     │    date     │ int64 │
├─────────────────┴─────────────┴───────┤
│                0 rows                 │
└───────────────────────────────────────┘

Row count and date span:
┌─────────┬────────────┬────────────┐
│  rows   │  min_date  │  max_date  │
│  int64  │    date    │    date    │
├─────────┼────────────┼────────────┤
│ 9841378 │ 2026-03-01 │ 2026-03-31 │
└─────────┴────────────┴────────────┘

Availability check:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┐
│ total_rows │ rows_with_gsc_data │
│   int64    │       int128       │
├────────────┼────────────────────┤
│    9841378 │            3611061 │
└────────────┴────────────────────┘



In [29]:
# Build 5 features + the label, using only the first half of March for features
agg = con.sql("""
    SELECT
        content_hash_id,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks ELSE 0 END) AS early_clicks,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS early_impressions,
        AVG(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_avg_position END) AS early_avg_position,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN ga4_engaged_sessions ELSE 0 END) AS early_engaged_sessions,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN scroll_events ELSE 0 END) AS early_scroll_events,
        SUM(CASE WHEN report_date > DATE '2026-03-15' THEN gsc_clicks ELSE 0 END) AS late_clicks
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

agg["is_declining"] = (agg["late_clicks"] < agg["early_clicks"]).astype(int)
print(agg["is_declining"].value_counts())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

is_declining
0    147749
1     28989
Name: count, dtype: int64


The trap: I added late_clicks as a feature, even though it is literally what my label is built from. This pushed the score from an honest 0.79 AUC to a perfect 1.0 AUC. A perfect score is not something to celebrate, it means the model was just given the answer directly. I removed late_clicks and kept the honest 0.79 AUC as my real result.

In [30]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

honest_features = ["early_clicks", "early_impressions", "early_avg_position",
                    "early_engaged_sessions", "early_scroll_events"]
X = agg[honest_features].fillna(0)
y = agg["is_declining"]

model = LogisticRegression(max_iter=1000)
model.fit(X, y)
honest_auc = roc_auc_score(y, model.predict_proba(X)[:, 1])
print("Honest AUC:", honest_auc)

# THE TRAP: late_clicks is literally what the label is built from — this is cheating on purpose
leaky_features = honest_features + ["late_clicks"]
X_leak = agg[leaky_features].fillna(0)
model_leak = LogisticRegression(max_iter=1000)
model_leak.fit(X_leak, y)
leaky_auc = roc_auc_score(y, model_leak.predict_proba(X_leak)[:, 1])
print("Leaky AUC (with the trap column):", leaky_auc)

Honest AUC: 0.794421222841298
Leaky AUC (with the trap column): 1.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data cannot tell me why a page declined, only that it did. Client history length differs a lot, ranging from clients starting as early as January 2025 to clients starting as late as June 2026, so some pages have much more past data than others. GA4 numbers are zero filled before a client's GA4 start date, so early zeros do not always mean zero engagement, sometimes it just means missing data. 63.3 percent of March rows had no usable search data at all, so my results only reflect the smaller group of pages that do. This slice only covers one month, so I cannot see longer seasonal patterns.

In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Back up the data-limits claims with real numbers

con.sql("""
    SELECT MIN(gsc_data_start) AS earliest_start, MAX(gsc_data_start) AS latest_start
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')
""").show()

┌────────────────┬──────────────┐
│ earliest_start │ latest_start │
│      date      │     date     │
├────────────────┼──────────────┤
│ 2025-01-27     │ 2026-06-02   │
└────────────────┴──────────────┘



## Self-check

Before you submit, confirm each line honestly:

- [yes] Every section above is filled — markdown thinking AND the code that backs it
- [yes] The notebook runs top to bottom with no errors (Runtime → Run all)
- [yes] No client names, URLs, or private queries anywhere
- [yes] My claims use careful words: observed, measured, directional, decision-support
- [yes] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.